# 03 -- Stage 3: Merge Image + Video Datasets

Combines `labels.csv` (Stage 1) and `dfdc_frame_manifest.csv`
(Stage 2) into a single manifest, then retrains on the union.

Reads/writes under `dataset_root` (Drive on Colab), not the local
session -- both input manifests and their referenced image/frame files
need to actually be on Drive already for this to find real data (frame
images specifically need transferring there manually from Kaggle; see
the project README's Stage 2 notes).

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import pandas as pd
from pathlib import Path
from config.loader import get_settings, resolve_dataset_root

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Dataset root:", dataset_root)

image_labels = pd.read_csv(dataset_root / settings.data.labels_csv)
frame_manifest_path = dataset_root / "dfdc_frame_manifest.csv"

if frame_manifest_path.exists():
    frame_labels = pd.read_csv(frame_manifest_path)
    # The manifest's filename column holds absolute paths from wherever it
    # was originally extracted (e.g. Kaggle's /kaggle/working/...) -- remap
    # to wherever you actually uploaded the frame images on Drive, matched
    # by filename alone (same approach as 02b's remap step).
    frames_dir = dataset_root / settings.data.synthetic_dir
    frame_labels["filename"] = frame_labels["filename"].apply(lambda p: str(frames_dir / Path(p).name))

    missing = (~frame_labels["filename"].apply(lambda p: Path(p).exists())).sum()
    if missing:
        print(f"[WARNING] {missing} of {len(frame_labels)} frame files not found under {frames_dir} -- "
              "check you uploaded the actual .jpg frames there, not just the manifest.")

    merged = pd.concat([image_labels, frame_labels], ignore_index=True)
    print(f"Merged {len(image_labels)} image rows + {len(frame_labels)} video frame rows")
else:
    merged = image_labels
    print(f"[WARNING] {frame_manifest_path} not found -- merging image data only. "
          "Transfer dfdc_frame_manifest.csv + its frame images to this path on Drive first.")

merged_path = dataset_root / "merged_labels.csv"
merged.to_csv(merged_path, index=False)
print("Merged manifest:", merged_path, "--", len(merged), "rows")

Rerun the Stage 1 training loop pointed at `merged_labels.csv`,
loading `models/image_model.pth` as the starting checkpoint.